# LAB 4.1 — Face Detection & Recognition with DeepFace
**Module 3 | Chitkara University | B.Tech AI Specialization**

> **Objective:** Build a face recognition system that registers 3 people, detects faces, and identifies them with ≥ 90% accuracy.

---
### Lab Workflow
1. Setup & Install Dependencies
2. Build Database / Capture Photos
3. Face Verification (same vs. different person)
4. Face Identification (who is this?)

## 📦 CELL 1 — Install Dependencies
Run this cell **once** to install required packages.

In [15]:
# Install required libraries
!pip install deepface opencv-python tf-keras --quiet

print("✅ Installation complete!")

✅ Installation complete!



[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔧 CELL 2 — Imports & Global Setup

In [16]:
import os
import cv2
import json
import time
import numpy as np
from IPython.display import display, Image as IPImage, clear_output
from deepface import DeepFace

# ── Global Config ──────────────────────────────────────────
DATABASE_PATH  = "./database"       # Folder holding one sub-folder per person
TEST_PATH      = "./test_images"    # Folder for accuracy evaluation
MODEL_NAME     = "VGG-Face"         # DeepFace backbone (VGG-Face is accurate & fast)
DISTANCE_METRIC = "cosine"          # Similarity metric
PEOPLE         = ["Taylor Swift", "Beyoncé", "Zendaya"]  # 3 registered people

# Create base folders
os.makedirs(DATABASE_PATH, exist_ok=True)
os.makedirs(TEST_PATH, exist_ok=True)
for person in PEOPLE:
    os.makedirs(os.path.join(DATABASE_PATH, person), exist_ok=True)
    os.makedirs(os.path.join(TEST_PATH, person), exist_ok=True)

print(f"✅ Database folder: {os.path.abspath(DATABASE_PATH)}")
print(f"✅ Test folder    : {os.path.abspath(TEST_PATH)}")
print(f"\nRegistered people: {PEOPLE}")
print("\nFolder structure created:")
for person in PEOPLE:
    print(f"  database/{person}/  ← add {person}1.jpg … {person}3.jpg here")

✅ Database folder: D:\EY-Assignments\module_3_cv_practice\Day 4\database
✅ Test folder    : D:\EY-Assignments\module_3_cv_practice\Day 4\test_images

Registered people: ['Taylor Swift', 'Beyoncé', 'Zendaya']

Folder structure created:
  database/Taylor Swift/  ← add Taylor Swift1.jpg … Taylor Swift3.jpg here
  database/Beyoncé/  ← add Beyoncé1.jpg … Beyoncé3.jpg here
  database/Zendaya/  ← add Zendaya1.jpg … Zendaya3.jpg here


## 📸 CELL 3 — SECTION 2: Capture Face Photos via Webcam

Run this for each person. Press **SPACE** to capture, **ESC** to stop early.

> ⚠️ **Skip this cell if you already have photos in `database/` folders.**

In [ ]:
def capture_face_photos(person_name: str, num_photos: int = 5):
    """
    Opens webcam and captures `num_photos` images for a given person.
    Saves them to database/<person_name>/
    Press SPACE to capture each photo, ESC to exit early.
    """
    save_dir = os.path.join(DATABASE_PATH, person_name)
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[ERROR] Cannot open webcam. Check connection.")
        return

    print(f"\n[INFO] Capturing {num_photos} photos for '{person_name}'")
    print("       Press SPACE to capture | ESC to stop early")

    count = 0
    while count < num_photos:
        ret, frame = cap.read()
        if not ret:
            break

        display_frame = frame.copy()
        cv2.putText(display_frame,
                    f"Person: {person_name} | Photo {count+1}/{num_photos}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(display_frame, "SPACE = capture | ESC = quit",
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
        cv2.imshow("Capture Face", display_frame)

        key = cv2.waitKey(1)
        if key == 27:    # ESC
            break
        elif key == 32:  # SPACE
            filename = os.path.join(save_dir, f"{person_name}{count+1}.jpg")
            cv2.imwrite(filename, frame)
            print(f"  ✓ Saved: {filename}")
            count += 1

    cap.release()
    cv2.destroyAllWindows()
    print(f"[INFO] Captured {count} photos for '{person_name}'")


# ── Uncomment to capture photos for each person ──────────────
for name in PEOPLE:
    capture_face_photos(name, num_photos=3)

# Or capture for a single person:
# capture_face_photos("alice", num_photos=5)

print("[INFO] Uncomment lines above to start capturing photos.")

## 🔍 CELL 4 — SECTION 3: Face Verification
**Are these two images the same person?**

In [27]:
def verify_two_faces(img_path1: str, img_path2: str):
    """
    Uses DeepFace.verify() to check if two images show the same person.
    Returns a dict with 'verified' (bool) and 'distance' (float).
    """
    print(f"\n[VERIFY] Comparing:")
    print(f"  Image 1: {img_path1}")
    print(f"  Image 2: {img_path2}")
    try:
        result = DeepFace.verify(
            img1_path   = img_path1,
            img2_path   = img_path2,
            model_name  = MODEL_NAME,
            distance_metric = DISTANCE_METRIC,
            enforce_detection = True
            
        )
        verdict = "✓ YES — Same Person" if result["verified"] else "✗ NO — Different Person"
        print(f"  Result     : {verdict}")
        print(f"  Distance   : {result['distance']:.4f}  (threshold: {result['threshold']:.4f})")
        print(f"  Model Used : {result['model']}")
        return result
    except Exception as e:
        print(f"  [ERROR] Verification failed: {e}")
        return None


# ── TEST 1: Same person (should return verified = True) ──────
# verify_two_faces("database/alice/alice1.jpg", "database/alice/alice2.jpg")
verify_two_faces("database/Zendaya/Zendaya1.png", "database/Zendaya/Zendaya3.png")


# ── TEST 2: Different person (should return verified = False) ─
verify_two_faces("database/Zendaya/Zendaya1.png", "database/Beyonce/Beyonce1.png")

# print("[INFO] Add image paths above and uncomment to run verification tests.")


[VERIFY] Comparing:
  Image 1: database/Zendaya/Zendaya1.png
  Image 2: database/Zendaya/Zendaya3.png
  Result     : ✓ YES — Same Person
  Distance   : 0.5398  (threshold: 0.6800)
  Model Used : VGG-Face

[VERIFY] Comparing:
  Image 1: database/Zendaya/Zendaya1.png
  Image 2: database/Beyonce/Beyonce1.png
  Result     : ✗ NO — Different Person
  Distance   : 0.8147  (threshold: 0.6800)
  Model Used : VGG-Face


{'verified': False,
 'distance': 0.814685,
 'threshold': 0.68,
 'confidence': 7.19,
 'model': 'VGG-Face',
 'detector_backend': 'opencv',
 'similarity_metric': 'cosine',
 'facial_areas': {'img1': {'x': 101,
   'y': 116,
   'w': 213,
   'h': 213,
   'left_eye': (245, 202),
   'right_eye': (166, 195)},
  'img2': {'x': 219,
   'y': 82,
   'w': 203,
   'h': 203,
   'left_eye': (355, 157),
   'right_eye': (277, 166)}},
 'time': 1.31}

## 🎯 CELL 5 — SECTION 4: Face Identification
**Who is this person? Search the entire database.**

In [28]:
def identify_face(query_image_path: str):
    """
    Uses DeepFace.find() to search the registered database for the best match.
    Returns: (person_name: str, distance: float)
    """
    print(f"\n[IDENTIFY] Searching database for: {query_image_path}")
    try:
        results_df = DeepFace.find(
            img_path    = query_image_path,
            db_path     = DATABASE_PATH,
            model_name  = MODEL_NAME,
            distance_metric = DISTANCE_METRIC,
            enforce_detection = True,
            silent      = True
        )

        if results_df and len(results_df[0]) > 0:
            top_match   = results_df[0].iloc[0]
            matched_path = top_match["identity"]
            person_name  = os.path.basename(os.path.dirname(matched_path))
            distance     = top_match["distance"]
            print(f"  ✅ Best Match : {person_name}")
            print(f"     Distance   : {distance:.4f}")
            print(f"     Matched to : {matched_path}")
            return person_name, distance
        else:
            print("  ❌ No match found in database.")
            return "Unknown", 1.0

    except Exception as e:
        print(f"  [ERROR] Identification failed: {e}")
        return "Unknown", 1.0


# ── Test identification ───────────────────────────────────────
identify_face("database/Taylor Swift/taylorSwift2.png")
# identify_face("test_images/bob/bob_test1.jpg")

# print("[INFO] Uncomment a line above to identify a face image.")


[IDENTIFY] Searching database for: database/Taylor Swift/taylorSwift2.png
26-02-28 17:24:06 - 🔴 Exception while extracting faces from ./database\Beyonce\Beyonce3.png: Face could not be detected in ./database\Beyonce\Beyonce3.png.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
  ✅ Best Match : Taylor Swift
     Distance   : 0.0952
     Matched to : ./database\Taylor Swift\taylorSwift2.png


('Taylor Swift', np.float64(0.095223))